# Notebook So Sánh Model

Notebook này chạy toàn bộ workflow cho ba kiến trúc `static`, `gru`, `hybrid`, sau đó gom metrics/report/backtest thành một bảng so sánh dùng trực tiếp cho báo cáo.

## 1. Cấu hình chạy

- `FORCE_RERUN = True` để train lại từ đầu.
- `RUN_MODEL_ARCHITECTURES` có thể rút gọn khi muốn chạy nhanh một model.
- Mỗi model được ghi vào một session riêng dưới `results/model_comparison_<timestamp>/`.

In [1]:
from __future__ import annotations

from datetime import datetime
import json
import os
from pathlib import Path
import re
import shutil
import subprocess
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT = PROJECT_ROOT.resolve()
os.chdir(PROJECT_ROOT)

src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from thesis.pipeline import run_pipeline
from thesis.shared.config import load_config
from thesis.shared.session_paths import configure_session_paths

CONFIG_PATH = PROJECT_ROOT / "config.toml"
RUN_MODEL_ARCHITECTURES = ["static", "gru", "hybrid"]
FORCE_RERUN = False
RUN_BACKTEST = True
RUN_REPORTING = True

# Stage to start from, matching main.py semantics:
# 1 = full workflow, 2 = skip data, 3 = skip data/features, 4 = train+backtest+report.
START_STAGE = 1

# Set True if you want the notebook to run scripts/data_download.py when raw data is missing.
DOWNLOAD_DATA_IF_MISSING = False

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
COMPARISON_DIR = PROJECT_ROOT / "results" / f"model_comparison_{RUN_ID}"
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

COMPARISON_DIR

/home/ultimatebrok/Downloads/thesis/.pixi/envs/default/lib/python3.13/site-packages/backtesting/_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

PosixPath('/home/ultimatebrok/Downloads/thesis/results/model_comparison_20260507_000004')

## 2. Helper tạo session và đọc kết quả

In [2]:
MODEL_LABELS = {
    "static": "Static LightGBM",
    "gru": "GRU-only",
    "hybrid": "Hybrid GRU + LightGBM",
}


def apply_stage_flags(config, start_stage: int) -> None:
    if start_stage > 1:
        config.workflow.run_data_pipeline = False
    if start_stage > 2:
        config.workflow.run_feature_engineering = False
    if start_stage > 3:
        config.workflow.run_label_generation = False
    if start_stage > 4:
        config.workflow.run_model_training = False
    if start_stage > 5:
        config.workflow.run_backtest = False


def write_variant_snapshot(session_dir: Path, architecture: str) -> None:
    snapshot = CONFIG_PATH.read_text()
    snapshot = re.sub(
        r'(?m)^architecture\s*=\s*"[^"]+"',
        f'architecture = "{architecture}"',
        snapshot,
        count=1,
    )
    (session_dir / "config" / "config_snapshot.toml").write_text(snapshot)


def ensure_required_inputs() -> None:
    config = load_config(CONFIG_PATH)
    raw_dir = PROJECT_ROOT / config.paths.data_raw
    raw_files = list(raw_dir.glob("*.parquet")) if raw_dir.exists() else []
    ohlcv_path = PROJECT_ROOT / config.paths.ohlcv
    features_path = PROJECT_ROOT / config.paths.features
    labels_path = PROJECT_ROOT / config.paths.labels

    if DOWNLOAD_DATA_IF_MISSING and START_STAGE <= 1 and not raw_files and not ohlcv_path.exists():
        subprocess.run([sys.executable, "scripts/data_download.py"], check=True)
        raw_files = list(raw_dir.glob("*.parquet")) if raw_dir.exists() else []

    if START_STAGE <= 1 and not raw_files and not ohlcv_path.exists():
        raise FileNotFoundError(
            "Không có raw parquet trong data/raw/XAUUSD và cũng chưa có "
            "data/processed/ohlcv.parquet. Hãy chạy `pixi run data` trước, "
            "hoặc đặt DOWNLOAD_DATA_IF_MISSING = True trong notebook."
        )
    if START_STAGE > 1 and not ohlcv_path.exists():
        raise FileNotFoundError(
            f"START_STAGE={START_STAGE} cần file {ohlcv_path.relative_to(PROJECT_ROOT)}. "
            "Hãy chạy từ START_STAGE=1 hoặc tạo OHLCV trước."
        )
    if START_STAGE > 2 and not features_path.exists():
        raise FileNotFoundError(
            f"START_STAGE={START_STAGE} cần file {features_path.relative_to(PROJECT_ROOT)}. "
            "Hãy chạy từ START_STAGE=2 hoặc tạo features trước."
        )
    if START_STAGE > 3 and not labels_path.exists():
        raise FileNotFoundError(
            f"START_STAGE={START_STAGE} cần file {labels_path.relative_to(PROJECT_ROOT)}. "
            "Hãy chạy từ START_STAGE=3 hoặc tạo labels trước."
        )


def prepare_config(architecture: str):
    config = load_config(CONFIG_PATH)
    config.model.architecture = architecture
    config.workflow.force_rerun = FORCE_RERUN
    config.workflow.run_backtest = RUN_BACKTEST
    config.workflow.run_reporting = RUN_REPORTING
    apply_stage_flags(config, START_STAGE)

    session_dir = COMPARISON_DIR / architecture
    configure_session_paths(config, session_dir)
    for subdir in ["config", "models", "predictions", "reports", "backtest", "logs"]:
        (session_dir / subdir).mkdir(parents=True, exist_ok=True)
    write_variant_snapshot(session_dir, architecture)
    config.workflow.session_timestamp = RUN_ID
    return config


def read_json(path: Path) -> dict:
    if not path.exists():
        return {}
    with path.open() as f:
        return json.load(f)


def flatten_metrics(architecture: str, session_dir: Path) -> dict:
    metrics = read_json(session_dir / "reports" / "model_metrics.json")
    wf = read_json(session_dir / "reports" / "walk_forward_history.json")
    bt = read_json(session_dir / "backtest" / "backtest_results.json").get("metrics", {})

    per_class = metrics.get("per_class", {})
    high_conf = metrics.get("high_confidence", {})
    return {
        "model": MODEL_LABELS.get(architecture, architecture),
        "architecture": architecture,
        "directional_accuracy": metrics.get("directional_accuracy"),
        "accuracy": metrics.get("accuracy"),
        "balanced_accuracy": metrics.get("balanced_accuracy"),
        "macro_f1": metrics.get("macro_f1"),
        "weighted_f1": metrics.get("weighted_f1"),
        "short_f1": per_class.get("Short", {}).get("f1"),
        "hold_f1": per_class.get("Hold", {}).get("f1"),
        "long_f1": per_class.get("Long", {}).get("f1"),
        "high_confidence_accuracy": high_conf.get("accuracy"),
        "high_confidence_count": high_conf.get("count"),
        "num_oof": metrics.get("total") or wf.get("total_oof_predictions"),
        "num_windows": wf.get("num_windows"),
        "backtest_return_pct": bt.get("return_pct"),
        "max_drawdown_pct": bt.get("max_drawdown_pct"),
        "sharpe_ratio": bt.get("sharpe_ratio"),
        "num_trades": bt.get("num_trades"),
        "session_dir": str(session_dir.relative_to(PROJECT_ROOT)),
    }


def sort_for_report(df: pd.DataFrame) -> pd.DataFrame:
    order = {"static": 0, "gru": 1, "hybrid": 2}
    return (
        df.assign(_order=df["architecture"].map(order).fillna(99))
        .sort_values("_order")
        .drop(columns="_order")
        .reset_index(drop=True)
    )

## 3. Chạy toàn bộ workflow

Cell này có thể mất lâu vì `static`, `gru`, và `hybrid` đều chạy walk-forward training.

In [3]:
ensure_required_inputs()

sessions: dict[str, Path] = {}

for architecture in RUN_MODEL_ARCHITECTURES:
    print(f"\n=== Running {MODEL_LABELS.get(architecture, architecture)} ===")
    config = prepare_config(architecture)
    run_pipeline(config)
    sessions[architecture] = Path(config.paths.session_dir)

sessions


=== Running Static LightGBM ===


  ⊘ SKIP Data Preparation: cached (ohlcv.parquet)

  ⊘ SKIP Feature Engineering: cached (features.parquet)

  ⊘ SKIP Label Generation: cached (labels.parquet)

────────────────────────────────────────   STAGE 4/6  ·  Model Training   ─────────────────────────────────────────

...

───────────────────────────────────────   STAGE 6/6  ·  Report Generation   ───────────────────────────────────────

Output()

Output()

──────────────────────────────────────────────── Pipeline Complete ────────────────────────────────────────────────

{'static': PosixPath('/home/ultimatebrok/Downloads/thesis/results/model_comparison_20260507_000004/static'),
 'gru': PosixPath('/home/ultimatebrok/Downloads/thesis/results/model_comparison_20260507_000004/gru'),
 'hybrid': PosixPath('/home/ultimatebrok/Downloads/thesis/results/model_comparison_20260507_000004/hybrid')}

## 4. Bảng so sánh tổng hợp

In [4]:
comparison_rows = [flatten_metrics(arch, session_dir) for arch, session_dir in sessions.items()]
comparison_df = sort_for_report(pd.DataFrame(comparison_rows))

output_csv = COMPARISON_DIR / "model_comparison_full.csv"
comparison_df.to_csv(output_csv, index=False)

display(comparison_df)
print(f"Saved: {output_csv.relative_to(PROJECT_ROOT)}")

,model,architecture,directional_accuracy,accuracy,balanced_accuracy,macro_f1,weighted_f1,short_f1,hold_f1,long_f1,high_confidence_accuracy,high_confidence_count,num_oof,num_windows,backtest_return_pct,max_drawdown_pct,sharpe_ratio,num_trades,session_dir
0,Static LightGBM,static,0.500250,0.416629,0.371601,0.362105,0.422782,0.446418,0.193502,0.446396,0.362500,80,35552,9,-2.973912,-5.701343,-0.269149,1063,results/model_comparison_20260507_000004/static
1,GRU-only,gru,0.502499,0.391215,0.351468,0.338502,0.405440,0.433598,0.153206,0.428703,0.000000,0,35129,9,0.000000,-0.000000,NaN,0,results/model_comparison_20260507_000004/gru
2,Hybrid GRU + LightGBM,hybrid,0.507960,0.425147,0.374416,0.366479,0.430810,0.428275,0.192010,0.479151,0.342857,35,35129,9,-7.022419,-7.815016,-0.678777,1044,results/model_comparison_20260507_000004/hybrid


Saved: results/model_comparison_20260507_000004/model_comparison_full.csv


## 5. Bảng rút gọn để đưa vào báo cáo

In [5]:
report_cols = [
    "model",
    "directional_accuracy",
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "weighted_f1",
    "short_f1",
    "hold_f1",
    "long_f1",
    "backtest_return_pct",
    "max_drawdown_pct",
    "sharpe_ratio",
]

report_df = comparison_df[report_cols]
display(report_df)
print(report_df.to_csv(index=False))

,model,directional_accuracy,accuracy,balanced_accuracy,macro_f1,weighted_f1,short_f1,hold_f1,long_f1,backtest_return_pct,max_drawdown_pct,sharpe_ratio
0,Static LightGBM,0.500250,0.416629,0.371601,0.362105,0.422782,0.446418,0.193502,0.446396,-2.973912,-5.701343,-0.269149
1,GRU-only,0.502499,0.391215,0.351468,0.338502,0.405440,0.433598,0.153206,0.428703,0.000000,-0.000000,NaN
2,Hybrid GRU + LightGBM,0.507960,0.425147,0.374416,0.366479,0.430810,0.428275,0.192010,0.479151,-7.022419,-7.815016,-0.678777


model,directional_accuracy,accuracy,balanced_accuracy,macro_f1,weighted_f1,short_f1,hold_f1,long_f1,backtest_return_pct,max_drawdown_pct,sharpe_ratio
Static LightGBM,0.5002500893176134,0.41662916291629165,0.3716006624484854,0.3621053847282459,0.42278201120271025,0.4464182646059129,0.19350215002388915,0.44639573955493567,-2.97391174902079,-5.701343245954837,-0.2691485531395737
GRU-only,0.5024989345627833,0.3912152352756981,0.3514675494801904,0.33850236335570916,0.4054397070723234,0.43359780389006064,0.15320582697453178,0.428703459202535,0.0,-0.0,
Hybrid GRU + LightGBM,0.507960413080895,0.4251473141848615,0.37441582526252937,0.3664785082336888,0.4308104955351987,0.4282747114304199,0.19200998751560547,0.4791508257550412,-7.022419157832009,-7.815016117089435,-0.6787769536606171

